In [ ]:
import pandas as pd
import numpy as np
import mlflow
import time
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_tracking_uri("sqlite:///../mlflow.db")  # File-based tracking is now in "maintenance mode", using SQLite instead
import pathlib
artifact_path = pathlib.Path('../mlruns').resolve().as_uri()
try:
    exp_id = mlflow.create_experiment('baseline_models', artifact_location=artifact_path)
except mlflow.exceptions.MlflowException:
    exp_id = mlflow.get_experiment_by_name('baseline_models').experiment_id
mlflow.set_experiment(experiment_id=exp_id)


In [ ]:
train_df = pd.read_json("../data/processed/train.jsonl", lines=True)
val_df = pd.read_json("../data/processed/val.jsonl", lines=True)

train_df["body_clean"] = train_df["body"].fillna("")
val_df["body_clean"] = val_df["body"].fillna("")

TASK_COLS = ["type", "queue", "category", "priority"]
print("Train:", len(train_df), "| Val:", len(val_df))

In [ ]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(train_df["body_clean"])
X_val = vectorizer.transform(val_df["body_clean"])
print("TF-IDF sekil:", X_train.shape)

In [ ]:
all_results = []

for task in TASK_COLS:
    y_train = train_df[task]
    y_val = val_df[task]

    models = {
        "LogReg": LogisticRegression(max_iter=1000),
        "LogReg_balanced": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "LinearSVC": LinearSVC(max_iter=2000),
        "LinearSVC_balanced": LinearSVC(max_iter=2000, class_weight="balanced"),
        "MultinomialNB": MultinomialNB(),
    }

    for name, model in models.items():
        with mlflow.start_run(run_name=f"{task}_{name}"):
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            acc = accuracy_score(y_val, preds)
            f1_macro = f1_score(y_val, preds, average="macro")
            f1_weighted = f1_score(y_val, preds, average="weighted")

            mlflow.log_param("task", task)
            mlflow.log_param("model", name)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_macro", f1_macro)
            mlflow.log_metric("f1_weighted", f1_weighted)

            labels_sorted = sorted(y_val.unique())
            cm = confusion_matrix(y_val, preds, labels=labels_sorted)
            fig, ax = plt.subplots(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                        xticklabels=labels_sorted, yticklabels=labels_sorted, ax=ax)
            plt.title(f"{task} - {name}")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            fig.savefig("cm_temp.png")
            mlflow.log_artifact("cm_temp.png")
            plt.close(fig)
            if os.path.exists('cm_temp.png'):
                os.remove('cm_temp.png')

        all_results.append({
            "task": task, "model": name,
            "accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted
        })

results_df = pd.DataFrame(all_results)

In [ ]:
pd.set_option("display.width", 120)
for task in TASK_COLS:
    print(f"=== {task} ===")
    print(results_df[results_df["task"] == task]
          .sort_values("f1_macro", ascending=False)
          .to_string(index=False))
    print()